# In-the-wild robustness with walking-frame features

Same Android sessions and session-loading pipeline as notebook 07, but
inference uses the 8-channel walking-frame CNN
(`models/cnn_wf_baseline.keras` from notebook 09). The 12-channel CNN
and 6-channel orientation-invariant CNN are kept as comparison
baselines.

In [4]:
import os, sys, json
from pathlib import Path
import numpy as np, pandas as pd
from ai_edge_litert.interpreter import Interpreter
import keras

_ROOT = Path('..').resolve(); sys.path.insert(0, str(_ROOT))
from utils.orientation_invariant_features import (
    compute_features, compute_walking_frame_features,
    ORIENTATION_INVARIANT_COLS, WALKING_FRAME_COLS,
)
ACT_LABELS = ['dws','ups','wlk','jog','std','sit']
G = 9.80665

labels_df = pd.read_csv('../../data/in_the_wild/labels.csv').set_index('session_dir')
print(labels_df[['activity','activity_id','pocket_orientation']])


            activity  activity_id                 pocket_orientation
session_dir                                                         
dws              dws            0                            natural
ups              ups            1                            natural
hod              wlk            2                            natural
hod2             wlk            2                            natural
hodanje          wlk            2                            natural
hodanje2         wlk            2                            natural
hodanje3         wlk            2                            natural
jog              jog            3                            natural
ED               wlk            2  screen-toward-thigh + upside-down
EG               wlk            2      screen-toward-thigh + upright
KD               wlk            2  camera-toward-thigh + upside-down
KG               wlk            2      camera-toward-thigh + upright


## Section: load_session — Android → iOS CoreMotion

Verified against the Sensor Logger app docs
([COORDINATES.md](https://github.com/tszheichoi/awesome-sensor-logger/blob/main/COORDINATES.md),
[CROSSPLATFORM.md](https://github.com/tszheichoi/awesome-sensor-logger/blob/main/CROSSPLATFORM.md))
and Apple's CMDeviceMotion documentation:

| Quantity | Android | iOS CoreMotion | Required transform |
|---|---|---|---|
| Gravity | m/s², face-up → (0, 0, +9.81) | g, face-up → (0, 0, −1) | `−android / 9.80665` (all three components) |
| User-acceleration | m/s² | g | `−android / 9.80665` (all three components) |
| Rotation rate | rad/s | rad/s | **identity** (gyroscope sign convention matches) |
| Attitude roll | rad | rad | **identity** (roll convention matches) |
| Attitude pitch | rad, CW about X positive | rad, CCW about X positive | sign flip |
| Attitude yaw | rad, CW positive (magnetic ref) | rad, CCW positive (true-north ref) | sign flip + relative offset |

In [5]:
def load_session(session_dir):
    base = Path('../../data') / session_dir
    df_ori  = pd.read_csv(base/'Orientation.csv').sort_values('time')
    df_grav = pd.read_csv(base/'Gravity.csv').sort_values('time')
    df_gyr  = pd.read_csv(base/'Gyroscope.csv').sort_values('time')
    df_tot  = pd.read_csv(base/'TotalAcceleration.csv').sort_values('time')

    df = pd.merge_asof(df_ori[['time','roll','pitch','yaw']],
                       df_grav[['time','x','y','z']], on='time', suffixes=('','_grav'))
    df = pd.merge_asof(df, df_gyr[['time','x','y','z']], on='time', suffixes=('','_gyro'))
    df = pd.merge_asof(df, df_tot[['time','x','y','z']], on='time', suffixes=('','_tot_acc'))
    df.columns = ['time','attitude.roll','attitude.pitch','attitude.yaw',
                  'raw_gravity.x','raw_gravity.y','raw_gravity.z',
                  'rotationRate.x','rotationRate.y','rotationRate.z',
                  'raw_total_acc.x','raw_total_acc.y','raw_total_acc.z']
    df['time_dt'] = pd.to_datetime(df['time'])
    df = df.set_index('time_dt').resample('20ms').mean(numeric_only=True).interpolate(method='linear').reset_index(drop=True)

    # Linear acceleration (Android does not report this directly when using
    # TotalAcceleration; we derive it as total - gravity, both in m/s^2).
    df['raw_linear_acc.x'] = df['raw_total_acc.x'] - df['raw_gravity.x']
    df['raw_linear_acc.y'] = df['raw_total_acc.y'] - df['raw_gravity.y']
    df['raw_linear_acc.z'] = df['raw_total_acc.z'] - df['raw_gravity.z']

    # Single sign flip + m/s^2 -> g  (Sensor Logger COORDINATES.md: all three
    # acceleration components differ in sign between Android and iOS).
    df['gravity.x'] = -df['raw_gravity.x'] / G
    df['gravity.y'] = -df['raw_gravity.y'] / G
    df['gravity.z'] = -df['raw_gravity.z'] / G
    df['userAcceleration.x'] = -df['raw_linear_acc.x'] / G
    df['userAcceleration.y'] = -df['raw_linear_acc.y'] / G
    df['userAcceleration.z'] = -df['raw_linear_acc.z'] / G

    # Rotation rate: identity (gyro sign convention matches iOS).
    # Roll: identity (roll convention matches iOS).
    # Pitch + yaw: sign flip (CROSSPLATFORM.md). Yaw further zeroed to the
    # session start because Android references magnetic north, iOS true north;
    # only relative yaw is portable without a magnetometer calibration.
    df['attitude.pitch'] = -df['attitude.pitch']
    df['attitude.yaw']   = -df['attitude.yaw']
    df['attitude.yaw']   = df['attitude.yaw'] - df['attitude.yaw'].iloc[0]

    cols = ['attitude.roll','attitude.pitch','attitude.yaw',
            'gravity.x','gravity.y','gravity.z',
            'rotationRate.x','rotationRate.y','rotationRate.z',
            'userAcceleration.x','userAcceleration.y','userAcceleration.z']
    return df[cols].iloc[150:-150].reset_index(drop=True)


def window(arr, w=128, s=64):
    if len(arr) < w: return np.empty((0, w, arr.shape[1]))
    return np.stack([arr[st:st+w] for st in range(0, len(arr)-w+1, s)], axis=0)


In [6]:
interp = Interpreter(model_path='../../models/cnn_best.tflite')
interp.allocate_tensors()
inp_idx, out_idx = interp.get_input_details()[0]['index'], interp.get_output_details()[0]['index']


def predict_12ch(X):
    Xn = X.copy().astype(np.float32)
    dyn = Xn[:,:,6:12]
    Xn[:,:,6:12] = (dyn - dyn.mean(axis=1, keepdims=True)) / (dyn.std(axis=1, keepdims=True) + 1e-8)
    out = np.zeros((len(Xn), 6), dtype=np.float32)
    for i, w in enumerate(Xn):
        interp.set_tensor(inp_idx, w[None])
        interp.invoke()
        out[i] = interp.get_tensor(out_idx)[0]
    return out


cnn_oinv = keras.models.load_model('../../models/cnn_oinv_baseline.keras')
with open('../../models/cnn_oinv_baseline.preproc.json') as f:
    oinv_meta = json.load(f)
DYN_OINV = oinv_meta['dynamic_channel_indices']


def predict_oinv(X):
    Xn = X.copy().astype(np.float32)
    dyn = Xn[:,:,DYN_OINV]
    Xn[:,:,DYN_OINV] = (dyn - dyn.mean(axis=1, keepdims=True)) / (dyn.std(axis=1, keepdims=True) + 1e-8)
    return cnn_oinv.predict(Xn, verbose=0)


cnn_wf = keras.models.load_model('../../models/cnn_wf_baseline.keras')


def predict_wf(X):
    Xn = X.copy().astype(np.float32)
    Xn = (Xn - Xn.mean(axis=1, keepdims=True)) / (Xn.std(axis=1, keepdims=True) + 1e-8)
    return cnn_wf.predict(Xn, verbose=0)


In [7]:
rows = []
for session, row in labels_df.iterrows():
    gt = int(row['activity_id'])
    df_raw = load_session(session)
    raw_np = df_raw.to_numpy()
    w12 = window(raw_np); n_win = len(w12)
    if n_win == 0:
        print(f'WARN: {session} too short'); continue

    feats_oinv = compute_features(df_raw, fs_hz=50.0, group_cols=None, keep_meta=False)
    wo = window(feats_oinv[ORIENTATION_INVARIANT_COLS].to_numpy())

    feats_wf = compute_walking_frame_features(df_raw, fs_hz=50.0, smooth_seconds=5.0,
                                              group_cols=None, keep_meta=False)
    ww = window(feats_wf[WALKING_FRAME_COLS].to_numpy())

    p12 = predict_12ch(w12).argmax(axis=1)
    po  = predict_oinv(wo).argmax(axis=1)
    pw  = predict_wf(ww).argmax(axis=1)

    c12 = float((p12 == gt).mean()); co = float((po == gt).mean()); cw = float((pw == gt).mean())
    rows.append({
        'session': session, 'orientation': row['pocket_orientation'],
        'true': ACT_LABELS[gt],
        '12ch_correct': c12,
        '6ch_correct': co,
        '8ch_wf_correct': cw,
        'n_windows': n_win,
    })
    print(f'{session:<22s} gt={ACT_LABELS[gt]:<4s}  '
          f'12ch={c12*100:5.1f}%   6ch_oinv={co*100:5.1f}%   8ch_wf={cw*100:5.1f}%')

res = pd.DataFrame(rows).set_index('session')


dws                    gt=dws   12ch=  0.0%   6ch_oinv=  0.0%   8ch_wf=100.0%
ups                    gt=ups   12ch=100.0%   6ch_oinv=100.0%   8ch_wf=100.0%
hod                    gt=wlk   12ch= 22.2%   6ch_oinv=  0.0%   8ch_wf=  0.0%
hod2                   gt=wlk   12ch=100.0%   6ch_oinv=100.0%   8ch_wf= 66.7%
hodanje                gt=wlk   12ch= 38.5%   6ch_oinv= 23.1%   8ch_wf= 46.2%
hodanje2               gt=wlk   12ch=  0.0%   6ch_oinv=  0.0%   8ch_wf=  0.0%
hodanje3               gt=wlk   12ch= 14.3%   6ch_oinv= 65.7%   8ch_wf= 57.1%
jog                    gt=jog   12ch=  0.0%   6ch_oinv=  0.0%   8ch_wf=100.0%
ED                     gt=wlk   12ch= 57.6%   6ch_oinv= 93.9%   8ch_wf= 78.8%
EG                     gt=wlk   12ch=100.0%   6ch_oinv=  0.0%   8ch_wf= 32.6%
KD                     gt=wlk   12ch=  0.0%   6ch_oinv=  0.0%   8ch_wf= 47.2%
KG                     gt=wlk   12ch= 76.9%   6ch_oinv= 82.1%   8ch_wf= 33.3%


In [8]:
def agg(col):
    weights = res['n_windows'].to_numpy(); fracs = res[col].to_numpy()
    return float(np.average(fracs, weights=weights)), float((fracs > 0.5).mean())


a12, s12 = agg('12ch_correct')
ao, so   = agg('6ch_correct')
aw, sw   = agg('8ch_wf_correct')

summary = pd.DataFrame({
    'window_acc':  [a12, ao, aw],
    'session_acc': [s12, so, sw],
}, index=['CNN-1D baseline (12ch raw)',
          'CNN-1D baseline (6ch orientation-invariant)',
          'CNN-1D baseline (8ch walking-frame, 1b)'])
print(summary.round(4).to_string())

print(f'\n8ch walking-frame vs. 12ch raw:    Δ window-acc = {aw-a12:+.4f}   Δ session-acc = {sw-s12:+.4f}')
print(f'8ch walking-frame vs. 6ch oinv:    Δ window-acc = {aw-ao:+.4f}   Δ session-acc = {sw-so:+.4f}')

os.makedirs('../results', exist_ok=True)
res.to_csv('../results/in_the_wild_walking_per_session.csv')
summary.to_csv('../results/in_the_wild_walking_summary.csv')


                                             window_acc  session_acc
CNN-1D baseline (12ch raw)                       0.4667       0.4167
CNN-1D baseline (6ch orientation-invariant)      0.4042       0.4167
CNN-1D baseline (8ch walking-frame, 1b)          0.4750       0.5000

8ch walking-frame vs. 12ch raw:    Δ window-acc = +0.0083   Δ session-acc = +0.0833
8ch walking-frame vs. 6ch oinv:    Δ window-acc = +0.0708   Δ session-acc = +0.0833


## Per-orientation breakdown (walking sessions only)

In [9]:
walk = res[res['true'] == 'wlk'].copy()
walk[['orientation', '12ch_correct', '6ch_correct', '8ch_wf_correct', 'n_windows']].round(3)


,orientation,12ch_correct,6ch_correct,8ch_wf_correct,n_windows
session,,,,,
hod,natural,0.222,0.000,0.000,9
hod2,natural,1.000,1.000,0.667,3
hodanje,natural,0.385,0.231,0.462,13
hodanje2,natural,0.000,0.000,0.000,13
hodanje3,natural,0.143,0.657,0.571,35
ED,screen-toward-thigh + upside-down,0.576,0.939,0.788,33
EG,screen-toward-thigh + upright,1.000,0.000,0.326,43
KD,camera-toward-thigh + upside-down,0.000,0.000,0.472,36
KG,camera-toward-thigh + upright,0.769,0.821,0.333,39


## Interpretation — the ranking flips vs MotionSense

**Headline.** On in-the-wild Android sessions the 8-channel walking-frame model is *the best of the three*: window-accuracy 0.475 vs 0.467 (12ch raw) vs 0.404 (6ch orientation-invariant), and session-accuracy 0.500 vs 0.417 vs 0.417. This is the **opposite ranking** to what MotionSense produced in notebook 09 (6ch 0.9473 > 12ch 0.9152 > 8ch wf 0.8956).

**Where the lift comes from — `dws` and `jog`.** Both classes had been failure modes for the other two baselines (12ch and 6ch both score 0.00 on `dws` and `jog`). The 8ch walking-frame model scores 1.00 on each. These are also the two activities where the body-frame projections (`a_f`, `a_s`, `gyro_v`) should help most: stair descent and jogging concentrate energy along the forward axis, which `a_f` exposes directly instead of folding into `acc_mag`.

**Orientation stress sessions (ED / EG / KD / KG).** All four are the same gait at different pocket orientations, so an orientation-invariant feature set should give them *consistent* per-window accuracies, even if the absolute number is low. The spreads:

| Model | ED | EG | KD | KG | max − min |
|---|---|---|---|---|---|
| 12ch raw | 0.576 | 1.000 | 0.000 | 0.769 | 1.000 |
| 6ch oinv | 0.939 | 0.000 | 0.000 | 0.821 | 0.939 |
| **8ch walking-frame** | **0.788** | **0.326** | **0.472** | **0.333** | **0.462** |

The walking-frame model's max accuracy is the lowest of the three, but its **spread is roughly half** — and KD ("camera-toward-thigh + upside-down"), which is 0.000 for *both* of the other models, is the only orientation any model handles non-trivially (0.472). That is the operative definition of orientation invariance: the failure mode at least becomes uniform across pocket placements rather than catastrophic at one of them.

**Why the 6ch model loses ground here while winning on MotionSense.** Notebook 07 already flagged this: `pitch_unwrapped` is not orientation-invariant by construction, and the gravity-projected channels `a_v` / `a_h` depend on Android's sensor-fusion gravity estimate, which differs from iOS CoreMotion (Stisen et al., 2015). The walking-frame channels are derived from a *smoothed* gravity vector and projected into a frame defined by the data itself (Mizell, 2003), which decouples the inference-time channels from the platform's gravity-fusion specifics.

**Decision**

- The 8-channel walking-frame representation is **not rejected**. It is the strongest representation on the only benchmark that actually tests deployment conditions (Android, varied pocket orientations).
- However, absolute window-accuracy of 0.475 is still well below anything deployable, and the natural-orientation walking sessions (`hod`, `hodanje`, `hodanje2`) show no improvement — so the walking-frame derivation as currently implemented helps with *orientation* variability but not with the platform-fusion / sampling-rate / sensor-noise gap.
- Notebook 11 (`walking-frame-v2`) should keep the body-frame projections but address the failure modes identified in notebook 09's interpretation: the all-dynamic Z-score that destroys the sit/std posture cue, and the 5 s smoothing window for the gravity estimate that is plausibly too long for stair classes.

> Stisen, A. et al. (2015). Smart devices are different: Assessing and mitigating mobile sensing heterogeneities for activity recognition. *SenSys 2015*, 127–140. https://doi.org/10.1145/2809695.2809718

> Mizell, D. (2003). Using gravity to estimate accelerometer orientation. *ISWC 2003*, 252–253. https://doi.org/10.1109/ISWC.2003.1241424